<a href="https://colab.research.google.com/github/JuliBakagianni/delta_meth/blob/main/run_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sentence_transformers
import transformers
import torch
import sklearn
import yaml

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())

---

4) Clone repository & checkout branch

Provide a `GITHUB_URL` variable to clone the repository into `/content/delta_meth`. Uncomment and set your GitHub URL as needed.

---

In [18]:
import os
GITHUB_URL = 'https://github.com/JuliBakagianni/delta_meth.git'
print('Cloning from', GITHUB_URL)
os.system('rm -rf /content/delta_meth')
os.system(f'git clone {GITHUB_URL} /content/delta_meth')

Cloning from https://github.com/JuliBakagianni/delta_meth.git


0

---

7) Write pipeline configuration (YAML/JSON)

Load and print the active configuration from `configs/config.yaml` in the cloned repo. If the file is not present, show a default config and write it to disk.

---

In [16]:
# Load config
import yaml
from pathlib import Path

cfg_path = Path('/content/delta_meth/configs/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
print('Loaded config:')
print(cfg)

Loaded config:
{'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'nli_model': 'joeddav/xlm-roberta-large-xnli', 'chunk_size': 150, 'similarity_threshold': 0.5, 'nli_threshold': 0.7, 'llm_model': 'openai/gpt-4o-mini'}


---

8) Implement pipeline runner and mock fallback

This cell imports the pipeline functions and attempts to run them. If heavy model imports fail, the notebook falls back to a mock runner that simulates embeddings and NLI labels for demonstration.

---

In [ ]:
# Run pipeline with fallback
import json
from pathlib import Path

# Ensure repo path is on sys.path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

mock_mode = False

# Try importing the real pipeline
from src.pipeline.run_pipeline import run_pipeline
from src.alignment.embeddings import encode_chunks
from src.nli.nli_model import predict_nli
print('Pipeline modules imported successfully; running real pipeline...')
contradiction, detailed = run_pipeline(config_path='/content/delta_meth/configs/config.yaml', verbose=True)
print(f"contradiction: {contradiction}")
if mock_mode:
    # Minimal mock implementation using local functions to simulate outputs
    from src.preprocessing.chunking import chunk_notes
    a = Path('/content/delta_meth/data/raw/dummy_note_20260212.txt')
    b = Path('/content/delta_meth/data/raw/dummy_note_20260213.txt')
    note_a = a.read_text(encoding='utf-8') if a.exists() else 'Patient no fever.'
    note_b = b.read_text(encoding='utf-8') if b.exists() else 'Patient has fever.'
    chunks_a = chunk_notes(note_a, chunk_size=50)
    chunks_b = chunk_notes(note_b, chunk_size=50)
    # Simulate alignment: pair each chunk i with j=i
    detailed = []
    for i, ca in enumerate(chunks_a):
        if i < len(chunks_b):
            cb = chunks_b[i]
        else:
            cb = chunks_b[-1] if chunks_b else ''
        # Mock sim score and NLI: if Greek words differ contain 'πυρετό' vs 'δεν' set contradiction
        sim_score = 0.6
        if 'πυρετό' in ca or 'πυρετό' in cb:
            label = 'contradiction'
            conf = 0.95
        else:
            label = 'neutral'
            conf = 0.6
        entry = {
            'i': i,
            'j': i if i < len(chunks_b) else len(chunks_b)-1,
            'sim_score': sim_score,
            'nli_label': label,
            'nli_confidence': conf,
            'chunk1': ca,
            'chunk2': cb,
        }
        detailed.append(entry)
    # Select highest-confidence contradiction
    contradictions = [d for d in detailed if d['nli_label']=='contradiction']
    contradiction = max(contradictions, key=lambda x: x['nli_confidence']) if contradictions else None

# Save results to results/run_output.json
out = {'contradiction': contradiction, 'detailed_pairs': detailed}
out_path = Path('/content/delta_meth/results')
out_path.mkdir(parents=True, exist_ok=True)
(out_path / 'run_output.json').write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nSaved run_output.json to /content/delta_meth/results/run_output.json')
print('Contradiction:', contradiction)
